In [1]:
# !pip install -q -U google-genai
# !pip install python-docx
# !pip install python-dotenv

# Detect Policy Changes

In [2]:
import os
from bs4 import BeautifulSoup
import requests
import time
import re
from datetime import datetime
import pandas as pd

import docx
from docx.shared import Pt, Inches

from google import genai
from google.genai import types

from itertools import zip_longest
from IPython.display import display, Markdown
from dotenv import load_dotenv

import pickle

import json

import docx
from docx.shared import Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH

load_dotenv(dotenv_path="local.env")

True

In [3]:
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Security Error: GEMINI_API_KEY environment variable is missing!")

In [4]:
# 1. Target URL config
TARGET_URL = "https://www.legislation.wa.gov.au/legislation/statutes.nsf/law_a517_currencies.html&view=consolidated"
BASE_URL = "https://www.legislation.wa.gov.au/legislation/statutes.nsf/"

# Create a local directory for storage
output_dir = "mining_act_html_versions"
os.makedirs(output_dir, exist_ok=True)

list_of_target_language = ['Spanish',
                              'Hindi',
                              'Tamil',
                              'French',
                              'German',
                               'Italian',
                               'Polish'
                              ]

## Fetching Mining Act 1978

In [5]:
print(f"Fetching page content from: {TARGET_URL}...")

try:
    # 2. Download the main versions index page
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    page_response = requests.get(TARGET_URL, headers=headers, timeout=15)
    page_response.raise_for_status()
    html_content = page_response.text

except Exception as e:
    print(f"Failed to fetch index page: {e}")
    exit()

soup = BeautifulSoup(html_content, "html.parser")

table = soup.find("table", class_="range compare")
if not table:
    print("Could not find the target legislation table on the live page.")
    exit()

rows = table.find("tbody").find_all("tr")
download_count = 0
skip_count = 0

print("\nStarting automated download of HTML act versions...")
print("-" * 60)

for row in rows:
    cells = row.find_all("td")
    if len(cells) < 4:
        continue

    # Extract clean naming tags from metadata columns
    act_name = cells[0].text.strip()            
    start_date = cells[1].text.strip().replace(" ", "_")  
    suffix = cells[3].text.strip()         

    # Filter out anchors that specifically direct to targeted document queries containing .htm
    html_link_node = row.find("a", href=lambda href: href and "query=" in href and ".htm" in href)

    if html_link_node:
        relative_href = html_link_node["href"]
        # Standardize matching URLs against the site's tag
        full_download_url = BASE_URL + relative_href
        filename = f"{act_name}_{suffix}_{start_date}.html".replace("/", "-")
        filepath = os.path.join(output_dir, filename)

        # CHECK: If the file already exists in output_dir, skip the download
        if os.path.exists(filepath):
            print(f"Skipping [{suffix}] ({cells[1].text.strip()}) -> Already exists locally.")
            skip_count += 1
            continue

        print(f"Downloading version tag [{suffix}] ({cells[1].text.strip()})...")

        try:
            file_response = requests.get(full_download_url, headers=headers, timeout=15)
            if file_response.status_code == 200:
                with open(filepath, "w", encoding="utf-8") as file:
                    file.write(file_response.text)
                download_count += 1
                print(f"  -> Saved successfully as: {filename}")
            else:
                print(f"  -> Download failed with HTTP Status: {file_response.status_code}")
        except Exception as e:
            print(f"  -> Connection error handling filename {filename}: {e}")

print("-" * 60)
print(f"Execution finished.")
print(f"Total files downloaded: {download_count}")
print(f"Total files skipped (already exist): {skip_count}")
print(f"Files are located in the '{output_dir}' directory.")

Fetching page content from: https://www.legislation.wa.gov.au/legislation/statutes.nsf/law_a517_currencies.html&view=consolidated...

Starting automated download of HTML act versions...
------------------------------------------------------------
Skipping [09-q0-00] (28 May 2026) -> Already exists locally.
Skipping [09-p0-00] (9 Sep 2025) -> Already exists locally.
Skipping [09-o0-00] (1 Sep 2025) -> Already exists locally.
Skipping [09-n0-00] (31 Aug 2025) -> Already exists locally.
Skipping [09-m0-00] (2 Jul 2025) -> Already exists locally.
Skipping [09-l0-00] (14 May 2024) -> Already exists locally.
Skipping [09-k0-01] (18 Nov 2023) -> Already exists locally.
Skipping [09-j0-01] (10 Aug 2023) -> Already exists locally.
Skipping [09-i0-00] (5 Apr 2023) -> Already exists locally.
Skipping [09-h0-00] (24 Mar 2023) -> Already exists locally.
Skipping [09-g0-00] (2 Nov 2022) -> Already exists locally.
Skipping [09-f0-00] (28 Sep 2022) -> Already exists locally.
Skipping [09-e0-00] (1 Jul

In [6]:
def get_file_metadata(file_path):
    """
    Parses a downloaded HTML file to extract its official currency start date 
    and version suffix from the text metadata.
    """
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            # Read just the first 5000 characters to find titles/metadata quickly
            head_content = f.read(5000)
            
        # 1. Fallback to extracting metadata directly from the filename if matching a pattern
        # Format assumed: "Mining Act 1978_Suffix_Date.html"
        filename = os.path.basename(file_path)
        match = re.search(r"Mining Act 1978_(.+?)_(.+?)\.html", filename)
        
        if match:
            suffix = match.group(1)
            date_str = match.group(2).replace("_", " ")
            # Handle "Current" or specific dates safely
            parsed_date = datetime.strptime(date_str, "%d %b %Y")
            return {"path": file_path, "date": parsed_date, "suffix": suffix}
            
    except Exception as e:
        print(f"Could not parse metadata for {os.path.basename(file_path)}: {e}")
    return None

def find_latest_versions(directory):
    """
    Scans the directory, gathers version metadata, and sorts files 
    chronologically to isolate the two newest records.
    """
    if not os.path.exists(directory):
        print(f"Directory '{directory}' does not exist.")
        return None, None

    all_versions = []
    
    # Loop through all downloaded files in the target directory
    for filename in os.listdir(directory):
        if filename.startswith("Mining Act 1978") and filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            metadata = get_file_metadata(file_path)
            if metadata:
                all_versions.append(metadata)
                
    if len(all_versions) < 2:
        print(f"Found {len(all_versions)} file(s). You need at least 2 files to compare.")
        return None, None

    # Sort files chronologically by their parsed currency date (newest first)
    all_versions.sort(key=lambda x: x["date"], reverse=True)
    
    # Isolate the two topmost records
    newest = all_versions[0]["path"]
    second_newest = all_versions[1]["path"]
    
    return newest, second_newest

In [7]:
def extract_clean_text(file_path):
    print(f"Cleaning raw markup from: {os.path.basename(file_path)}...")
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")
    
    content_area = soup.find("section", id="main") or soup.find("section", id="outer")
    target_soup = content_area if content_area else soup
    for element in target_soup(["script", "style", "meta", "link", "header", "footer", "nav", "form"]):
        element.decompose()
    text = target_soup.get_text(separator="\n")
    
    cleaned_lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(cleaned_lines)

def split_into_chunks(text, chunk_size=100000, overlap = 100000*0.2):
    """
    Splits text by lines to avoid cutting a section or sentence in half.
    Includes an overlap of approximately `overlap` characters between chunks.
    """
    if overlap >= chunk_size:
        raise ValueError("Overlap size must be smaller than chunk size.")
        
    chunks = []
    lines = text.splitlines()
    
    current_chunk = []
    current_size = 0
    new_data_added = False  
    
    for line in lines:
        current_chunk.append(line)
        current_size += len(line) + 1 
        new_data_added = True
        
        if current_size >= chunk_size:
            chunks.append("\n".join(current_chunk))
            
            overlap_chunk = []
            overlap_size = 0
            for l in reversed(current_chunk):
                if overlap_size + len(l) + 1 > overlap and overlap_chunk:
                    break
                overlap_chunk.insert(0, l)
                overlap_size += len(l) + 1
            
            current_chunk = overlap_chunk
            current_size = overlap_size
            new_data_added = False 
            
    if current_chunk and new_data_added:
        chunks.append("\n".join(current_chunk))

    return chunks

def extract_date_from_filename(filename):
    """
    Helper function to pull the clean date string from the filename 
    e.g., 'Mining Act 1978_09-p0-00_9_Sep_2025.html' -> '9_Sep_2025'
    """
    match = re.search(r"Mining Act 1978_.+?_(.+?)\.html", os.path.basename(filename))
    return match.group(1) if match else "UnknownDate"

def save_markdown_to_docx(markdown_text, output_filename):
    doc = docx.Document()
    
    # Configure base style constraints
    style = doc.styles['Normal']
    font = style.font
    font.name = 'Arial'
    font.size = Pt(11)
    
    print(f"Generating formatted report configuration...")
    lines = markdown_text.split('\n')
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
            
        if stripped.startswith('# '):
            doc.add_heading(stripped[2:], level=1)
        elif stripped.startswith('## '):
            doc.add_heading(stripped[3:], level=2)
        elif stripped.startswith('### '):
            doc.add_heading(stripped[4:], level=3)
        elif stripped.startswith('* ') or stripped.startswith('- '):
            cleaned_bullet = re.sub(r'\*\*(.*?)\*\*', r'\1', stripped[2:])
            doc.add_paragraph(cleaned_bullet, style='List Bullet')
        else:
            cleaned_text = re.sub(r'\*\*(.*?)\*\*', r'\1', line)
            doc.add_paragraph(cleaned_text)
            
    doc.save(output_filename)
    print(f"Success! Document created at: {os.path.abspath(output_filename)}")

In [8]:
# new_mining_act_file = "Mining Act 1978_09-p0-00_9_Sep_2025.html" 
# old_mining_act_file = "Mining Act 1978_09-o0-00_1_Sep_2025.html"


OUTPUT_DIR = "mining_act_html_versions"

# model= "gemma-4-31b-it"
model = "gemini-3.1-flash-lite"

max_retries = 5

new_mining_act_file, old_mining_act_file = find_latest_versions(OUTPUT_DIR)

print(f"New: {new_mining_act_file}")
print(f"Old: {old_mining_act_file}")
old_mining_act_file

New: mining_act_html_versions/Mining Act 1978_09-q0-00_28_May_2026.html
Old: mining_act_html_versions/Mining Act 1978_09-p0-00_9_Sep_2025.html


'mining_act_html_versions/Mining Act 1978_09-p0-00_9_Sep_2025.html'

In [9]:
# new_file_path = os.path.join(OUTPUT_DIR, new_mining_act_file)
# old_file_path = os.path.join(OUTPUT_DIR, old_mining_act_file)

old_text_content = extract_clean_text(old_mining_act_file)
new_text_content = extract_clean_text(new_mining_act_file)

print("\nChunking text into manageable segments...")
old_chunks = split_into_chunks(old_text_content)
new_chunks = split_into_chunks(new_text_content)

if not os.path.exists(new_mining_act_file) or not os.path.exists(old_mining_act_file):
    print("Error: One or both of the specified Mining Act HTML files are missing.")
    exit()

print("Reading HTML source files...")
with open(old_mining_act_file, "r", encoding="utf-8") as f:
    old_html_content = f.read()

with open(new_mining_act_file, "r", encoding="utf-8") as f:
    new_html_content = f.read()

Cleaning raw markup from: Mining Act 1978_09-p0-00_9_Sep_2025.html...
Cleaning raw markup from: Mining Act 1978_09-q0-00_28_May_2026.html...

Chunking text into manageable segments...
Reading HTML source files...


In [10]:
def evaluate_block_diff(block_report: str, 
                        old_raw_chunk: str, 
                        new_raw_chunk: str) -> dict:
    """
    Audits a single chunk's diff report against its raw source chunks 
    to ensure changes were accurately extracted without hallucinations.
    """
    eval_model = 'gemini-3.1-flash-lite'
    
    audit_prompt = f"""
    You are a high-level legal document auditor. Your job is to verify if a localized [DIFF REPORT] 
    accurately reflects the changes between an [OLD TEXT CHUNK] and a [NEW TEXT CHUNK].

    Analyze the [DIFF REPORT] and evaluate it based on two strict criteria:
    1. **Hallucination Detection**: Does the report claim a section was added, deleted, or altered when the raw text shows no such change?
    2. **Omission Detection**: Are there blatant text differences between the OLD and NEW chunks that the report completely missed?

    Return your audit analysis strictly as a JSON object with this exact schema:
    {{
       "contains_hallucinations": true or false,
       "missed_significant_changes": true or false,
       "hallucinated_details": ["list of false claims made in the report, empty if none"],
       "missed_details": ["list of actual text differences skipped by the report, empty if none"],
       "block_quality_score": 0.0 to 1.0
    }}

    [OLD TEXT CHUNK]
    {old_raw_chunk}

    [NEW TEXT CHUNK]
    {new_raw_chunk}

    [DIFF REPORT]
    {block_report}
    """
    
    try:
        response = client.models.generate_content(
            model=eval_model,
            contents=audit_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"  └─ [EVAL ERROR] Could not audit block: {e}")
        return {
            "contains_hallucinations": False, 
            "missed_significant_changes": False, 
            "block_quality_score": 1.0
        }
        
def compare_new_old_chunks(new_chunk, old_chunk, 
                          new_mining_act_file,old_mining_act_file):
    respone_text = """"""
    block_scores = []
    
    for idx, (old_chunk, new_chunk) in enumerate(zip_longest(old_chunks, new_chunks, fillvalue="")):
        
        if old_chunk == new_chunk:
            print(f"Block {idx + 1}: No text variations found. Skipping API call.")
            continue
            
        print(f"Block {idx + 1}: Text variation detected! Requesting localized analysis...")
        
        prompt = f"""
        You are an expert legislative data analyst. 
        Compare the following text block from the older version of the Mining Act 1978 against the matching block from the newer version.
    
        # Please provide a structured report detailing:
        # 1. **Executive Summary**: A brief overview of how extensive the changes are.
        # 2. **Specific Section Amendments**: Detail which sections, parts, or schedules were added, deleted, or altered. 
        # 3. **Textual Differences**: Highlight significant changes in phrasing, definitions, or penalties.
    
        ---
        [OLD REVISION TEXT SEGMENT FROM {old_mining_act_file}]
        {old_chunk}
        ---
        [NEW REVISION TEXT SEGMENT FROM {new_mining_act_file}]
        {new_chunk}
        ---
        """
        
        try:
            # Generate localized diff report
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                )
            )
            
            # Display the findings for this block
            print(f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---")
            print(response.text)
            respone_text += f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---"

            # # ─── ADD THE BLOCK-LEVEL EVALUATION HERE ───
            # print(f"  └─ [RUNNING EVALUATION] Validating Block {idx + 1} precision...")
            # evaluation = evaluate_block_diff(
            #     block_report=response.text,
            #     old_raw_chunk=old_chunk,
            #     new_raw_chunk=new_chunk
            # )
            
            # score = evaluation.get("block_quality_score", 1.0)
            # block_scores.append(score)
            # print(f"  └─ [EVAL RESULT] Quality Score: {score:.2%}")
            
            # if evaluation.get("contains_hallucinations"):
            #     print(f"     ⚠️ WARNING: Hallucinations detected: {evaluation['hallucinated_details']}")
            # if evaluation.get("missed_significant_changes"):
            #     print(f"     ⚠️ WARNING: Missed changes: {evaluation['missed_details']}")
            # # ───────────────────────────────────────────
            
            respone_text += response.text
            print("-" * 60)
            
            # Pause briefly to prevent hitting requests-per-minute limits
            time.sleep(120)
            
        except Exception as e:
            max_retries = 3
            retry_delay = 120
            success = False
            
            for attempt in range(max_retries):
                try:
                    time.sleep(120)
                    print(f"Retrying in 120 seconds...")
                    # Generate localized diff report
                    response = client.models.generate_content(
                        model=model,
                        contents=prompt,
                        config=types.GenerateContentConfig(
                            temperature=0.0,
                        )
                    )
                    
                    # Display the findings for this block
                    print(f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---")
                    print(response.text)
                    respone_text += f"\n--- [DIFF REPORT FOR BLOCK {idx + 1}] ---"
                    respone_text += response.text
                    print("-" * 60)
                    
                    success = True
                    time.sleep(120)
                    break  # Exit the retry loop on success
                    
                except Exception as e:
                    # Check if it's a server error (500, 503, etc.)
                    print(f"Attempt {attempt + 1} failed for Block {idx + 1}: {e}")
                    if attempt < max_retries - 1:
                        print(f"Retrying in {retry_delay} seconds...")
                        time.sleep(retry_delay)
                    else:
                        print(f"Block {idx + 1} permanently failed after {max_retries} attempts. Skipping to next block.")
                        
            # Keep the script moving even if one block errors out completely
            if not success:
                with open(os.path.join(OUTPUT_DIR, "failed_blocks.log"), "a") as log:
                    log.write(f"Block {idx + 1} failed with error.\n")
                    
        # ─── ADD THE BLOCK-LEVEL EVALUATION HERE ───
        print(f"  └─ [RUNNING EVALUATION] Validating Block {idx + 1} precision...")
        evaluation = evaluate_block_diff(
            block_report=response.text,
            old_raw_chunk=old_chunk,
            new_raw_chunk=new_chunk
        )
        
        score = evaluation.get("block_quality_score", 1.0)
        block_scores.append(score)
        print(f"  └─ [EVAL RESULT] Quality Score: {score:.2%}")
        
        if evaluation.get("contains_hallucinations"):
            print(f"     ⚠️ WARNING: Hallucinations detected: {evaluation['hallucinated_details']}")
        if evaluation.get("missed_significant_changes"):
            print(f"     ⚠️ WARNING: Missed changes: {evaluation['missed_details']}")
            # ───────────────────────────────────────────
    print("\nAutomated chunked audit finished successfully.")

    return respone_text, block_scores

def evaluate_faithfulness(master_report: str, source_diff_context: str) -> float:
    """
    Evaluates whether the master report contains any hallucinations 
    based strictly on the source text blocks.
    """
    # Use your existing client initialized via client = genai.Client()
    # Using a capable reasoning/extraction model like gemini-2.5-pro or gemini-2.5-flash
    eval_model = "gemini-3.1-flash-lite" 
    
    # --- PHASE 1: CLAIM EXTRACTION ---
    extraction_prompt = f"""
    Analyze the following Legislative Reform Audit Report. 
    Extract every single granular factual claim, assertion, or legal change statement made in the text.
    Return the output STRICTLY as a JSON list of strings. Do not include markdown formatting or commentary.
    
    [REPORT]
    {master_report}
    """
    
    try:
        extraction_res = client.models.generate_content(
            model=eval_model,
            contents=extraction_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        claims = json.loads(extraction_res.text)
    except Exception as e:
        print(f"Error extracting claims during evaluation: {e}")
        return 0.0

    if not claims:
        print("No claims extracted from the report to evaluate.")
        return 1.0

    # --- PHASE 2: CLAIM VERIFICATION ---
    verification_prompt = f"""
    You are a strict legal auditor. Your job is to check if a list of claims is completely supported by the [VERIFIED SOURCE CONTEXT] below.
    
    For each claim in the list, determine if it is:
    - "FAITHFUL": The claim is fully and explicitly supported by the context.
    - "HALLUCINATION": The claim contradicts the context, adds outside information, or speculates on facts not explicitly written.
    
    Return your evaluation strictly as a JSON object with the following structure:
    {{
       "analysis": [
          {{"claim": "the string of the claim", "status": "FAITHFUL" or "HALLUCINATION", "reason": "brief reason"}}
       ]
    }}

    [VERIFIED SOURCE CONTEXT]
    {source_diff_context}
    
    [CLAIMS TO EVALUATE]
    {json.dumps(claims)}
    """
    
    try:
        verification_res = client.models.generate_content(
            model=eval_model,
            contents=verification_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        results = json.loads(verification_res.text)
        
        # Calculate Faithfulness Score: (Faithful Claims / Total Claims)
        analysis_list = results.get("analysis", [])
        total_claims = len(analysis_list)
        faithful_claims = sum(1 for item in analysis_list if item["status"] == "FAITHFUL")
        
        faithfulness_score = faithful_claims / total_claims if total_claims > 0 else 0.0
        
        print(f"\n===== FAITHFULNESS METRIC CRITIQUE =====")
        print(f"Total Extracted Claims Checked: {total_claims}")
        print(f"Faithful Assertions: {faithful_claims}")
        print(f" Hallucinations Detected: {total_claims - faithful_claims}")
        print(f"--> Faithfulness Score: {faithfulness_score:.2%}")
        
        # Print failures for clear debugging
        if faithful_claims < total_claims:
            print("\nDetected Hallucinations:")
            for item in analysis_list:
                if item["status"] == "HALLUCINATION":
                    print(f"  - [FAILED]: {item['claim']}")
                    print(f"    Reason: {item['reason']}")
                    
        return faithfulness_score

    except Exception as e:
        print(f"Error running claim verification: {e}")
        return 0.0


def get_policy_changes(respone_text,
                      new_mining_act_file,
                      old_mining_act_file):

    new_date_tag = extract_date_from_filename(new_mining_act_file) 
    old_date_tag = extract_date_from_filename(old_mining_act_file)
    
    master_prompt = f"""
    You are an expert legal data analyst, legislative draftsman, and corporate risk auditor. 
    Your task is to synthesize block-by-block difference data into a definitive, 
    executive-ready "Legislative Reform Audit Report" tracking structural, 
    administrative, and legal modifications to the Western Australian Mining Act 1978.

    ---
    ### CRITICAL EXECUTION RULES:
    1. STRICT ADHERENCE: Base the entire report ONLY on the factual text provided within the [SOURCE DIFF REPORTS] block below. Do not speculate, invent, extrapolate, or introduce historical knowledge not present in the text.
    2. CLEAN OUTPUT CONSTRAINT: Output raw, clean Markdown text starting directly with the title. Do not include any conversational introductions, greetings, meta-commentary, notes, or concluding pleasantries.
    3. CITATION STRIPPING: Do not include any block labels, inline citation strings, or bracketed source tags (e.g., do not write "[Block 1]" or "[From CONTEXT]").
    4. FORMATTING RULES: Use structural Markdown elements carefully. Ensure table columns are cleanly aligned. Avoid large text walls by utilizing bold keywords and scannable bullet configurations.
    5. NO OPERATIONAL INFERENCE: Do not invent or assume administrative motives. For example, if a section is noted as omitted or deleted, state exactly what was deleted without adding phrases like "reflecting a consolidation of oversight" unless the source text explicitly states that exact reason.
    6. TONE MATCHING: Avoid upgrading the scale of the amendments. If the source documents characterize updates as "minimal," "targeted," or "housekeeping," do not describe them in your summary as a "strategic evolution" or a "proactive shift toward a centralized model." Match the exact modesty of the source text.
    7. DYNAMIC LAYOUT TRUNCATION: If the [SOURCE DIFF REPORTS] contain zero mentions of specific sections listed in this prompt template layout (such as Section 20, Section 40D, Section 63A, or Part 4AA approvals), you MUST completely delete those headings and sub-headings from the final output. 
    Do not include placeholder statements like "No data found for Section 20" or "No changes were noted."
    ---

    # Legislative Reform Audit Report: MINING ACT 1978
    **Comparative Analysis: Revision {old_date_tag} to Revision {new_date_tag}**

    ## 1. Executive Summary
    Provide a high-level corporate and legal synthesis of the transition between these two historical states. 
    Contrast the structural shift from a fragmented, legacy framework (where environmental and operational compliance 
    was managed passively through ad-hoc conditions tied to individual tenement leases) into a unified, 
    proactive, and centralized baseline regulatory architecture.

    ## 2. Definitive Statutory Matrix
    Consolidate all observed adjustments into clear, itemized groupings based on their exact operational outcome:

    ### A. Major Part and Section Additions
    List and analyze all entirely new legislative mechanisms introduced in the new revision. 
    Detail the core structural components and operational requirements for:
    * **Part 4AA (Conditions and Approvals):** Detail the operational mechanisms of Divisions 1 through 7, 
    explicitly capturing concepts like Eligible Mining Activities (EMA), EMA Notices, Excluded Area Notices, 
    and Centralized Programmes of Work.
    * **Sections 40A & 40B:** Break down how these sections redefine rights relative to 
    "available land" versus "conservation land."
    * **Sections 103AI through 103AV:** Detail the compliance boundaries governing Programmes of Work, 
    Mining Development and Closure Proposals, Mine Closure Plans, and Environmental Securities.

    ### B. Repealed, Omitted, and Deleted Provisions
    List all standalone provisions completely removed from the text, including Sections 46(aa), 46A, 52(1a), 60(1a), 63(aa), 63AA, 70I, 70P, 82A, 84, and 84AA (or any mineral-specific items like Section 8A).
    * **Regulatory Transition:** For each deletion, define what administrative oversight or compliance mechanism was detached from individual tenement lease clauses to be migrated into the unified framework.

    ### C. Altered and Modernized Sections
    Identify pre-existing sections undergoing cross-reference updates or semantic rewrites (referencing Sections 6, 8, 20, 40C, 40D, 55A, 63A, 69D, 70C, 70F, 70H, 70IA, 74, 82, 84A, 90, 96, 102A, 126, 139, 140, 159, and 162). Frame these updates strictly around administrative simplification and cross-referencing alignment with Part 4AA.

    ## 3. Structural and Thematic Analysis

    ### A. Formal Modifications to Legal Terminology
    Construct a markdown table detailing exact shifts in legal syntax used to alter enforcement dynamics. Follow this exact format:
    | Category / Context | Legacy Phrasing | Modernized / Updated Phrasing | Strategic Regulatory Impact |
    | :--- | :--- | :--- | :--- |

    Ensure the table explicitly incorporates tracking for:
    1. The shift from presumptive status ("Deemed") to mandatory status ("Taken").
    2. The expansion from standard "Mining Proposal" documentation to integrated "Mining Development and Closure Proposal" compliance.
    3. The shift from gender-specific oversight ("he deems") to formal institutional delegation ("the Governor thinks").

    ### B. Expansion of Liability Boundaries
    Contrast the framing of liability limitations explicitly documented within Section 20 and Section 40D. Detail the legal and financial implications of shifting from narrow protections targeting "fire damage to trees" to an expanded, omnidirectional standard covering "damage or injury to property or livestock whether resulting from fire... or any other cause."

    ### C. Compliance Architecture, Punitive Actions, and Forfeiture Triggers
    * **Forfeiture Modifications:** Map out how the administrative triggers for legal forfeiture under Section 63A and Section 96 have expanded to enforce compliance with the new Part 4AA standards.
    * **Remediation Binding:** Detail the operational execution of Section 103AV and Section 139/140, clarifying how environmental remediation is tied directly to financial securities and enhanced enforcement metrics.

    ## 4. Administrative and Historical Metadata
    Summarize the structural updates regarding the Act's layout, specifically tracking table formatting transitions, the consolidation of chronological amendment tracking (covering 1987 to 2025 tables), the status of uncommenced provisions tables, and the technical relocation of savings/transitional provisions to secondary notes sections.

    ---
    ### [SOURCE DIFF REPORTS]
    {respone_text}
    """
    
    master_response = client.models.generate_content(model='gemini-3.1-flash-lite',
                                                        contents=master_prompt,
                                                            config=types.GenerateContentConfig(temperature=0.0)
                                                        )

    faithfulness_rating = evaluate_faithfulness(master_report=master_response.text, 
                                                source_diff_context=respone_text
                                                )
    
    # You can choose to raise an alert or halt translation if the score drops below 95%
    if faithfulness_rating < 0.95:
        print("⚠️ WARNING: Report fell below safety threshold for legal faithfulness.")
    # ───────────────────────────────
    
    dynamic_filename = f"Legislative_Reform_Audit_Report_{old_date_tag}_to_{new_date_tag}.docx"
    
    save_markdown_to_docx(master_response.text, dynamic_filename)

    return master_response, new_date_tag, old_date_tag, faithfulness_rating

def evaluate_translation_compliance(original_english_text: str, 
                                    translated_text: str, 
                                    target_language: str) -> dict:
    """
    Evaluates translation quality, legal entity preservation, and formatting compliance.
    """
    eval_model = "gemini-3.1-flash-lite"
    
    compliance_prompt = f"""
    You are an expert international legal auditor fluent in English and formal {target_language} legal terminology.
    Your job is to critically evaluate a translated [LEGAL REPORT] against its [ORIGINAL ENGLISH SOURCE].

    Evaluate the translation based on three criteria:
    1. **Legal Entity Preservation**: Are proper nouns, alphanumeric codes, and statutory references (e.g., "Mining Act 1978", "Part 4AA", "Section 103AV") left exactly in their original alphanumeric format?
    2. **Structural Integrity**: Did the translation maintain the exact markdown layout, tables, bullet configurations, and headings (#, ##, ###)?
    3. **Translation Fidelity**: Is the translation professional, high-level legal {target_language} without mistranslations or omissions?

    Return your audit analysis strictly as a JSON object with this exact schema:
    {{
       "altered_statutory_codes": true or false,
       "broken_markdown_structures": true or false,
       "translation_quality_score": 0.0 to 1.0,
       "errors_detected": ["list of specific layout, code, or translation errors found, empty if none"]
    }}

    [ORIGINAL ENGLISH SOURCE]
    {original_english_text}

    [LEGAL REPORT TRANSLATION]
    {translated_text}
    """
    
    try:
        response = client.models.generate_content(
            model=eval_model,
            contents=compliance_prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0.0
            )
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"  └─ [EVAL ERROR] Translation evaluation failed: {e}")
        return {
            "altered_statutory_codes": False,
            "broken_markdown_structures": False,
            "translation_quality_score": 1.0,
            "errors_detected": []
        }

def tranlate_master_respone(master_response_text,target_language,
                           new_date_tag, old_date_tag):

    master_response_tranlate = f"""
                                You are an expert legal translator and legislative archivist fluent in both 
                                English and formal {target_language} legal terminology.
    
                                Your task is to translate the provided English "Legislative Reform Audit Report" 
                                into professional, high-level legal {target_language}.
                                
                                ---
                                ### Strict Content & Formatting Guidelines:
                                - Do not interpret, expand, or summarize the text. 
                                Translate the technical legal content exactly as written.
                                - Maintain the exact markdown layout, including headings (#, ##, ###), bullet lists, 
                                and markdown table structures.
                                - Keep proper nouns, citations, alphanumeric codes, 
                                and official statutory references (e.g., "Mining Act 1978", "Part 4AA", "Section 103AV") 
                                in their original format unless a direct, universally accepted legal equivalent exists in {target_language}.
                                - Ensure the final output is completely clean and formatted entirely in Markdown. 
                                Do not include any conversational introductions, meta-commentary, or closing remarks.
                                ---
                                
                                ### REPORT TO TRANSLATE:
                                {master_response_text}
                                """
    
    master_response_tranlate = client.models.generate_content(model='gemini-3.1-flash-lite',
                                                            contents=master_response_tranlate,
                                                            config=types.GenerateContentConfig(temperature=0.0)
                                                        )
    display(Markdown(master_response_tranlate.text))

    # 3. ─── DEPLOY TRANSLATION COMPLIANCE CHECKS ───
    print(f"🌐 Running translation quality checks for {target_language}...")
    eval_results = evaluate_translation_compliance(original_english_text=master_response_text,
                                                    translated_text=master_response_tranlate,
                                                    target_language=target_language)
    
    print(f"  └─ [RESULT] Quality Score: {eval_results['translation_quality_score']:.2%}")
    
    if eval_results.get("altered_statutory_codes"):
        print(f"     ⚠️ CRITICAL WARNING: Statutory codes or section numbers were altered in translation!")
    if eval_results.get("broken_markdown_structures"):
        print(f"     ⚠️ WARNING: Markdown table or heading hierarchy broke during translation.")
    if eval_results.get("errors_detected"):
        print(f"     📋 Logged Discrepancies: {eval_results['errors_detected']}")
    # ───────────────────────────────────────────────

    save_markdown_to_docx(master_response_tranlate.text, 
                          f"Legislative_Reform_Audit_Report_{old_date_tag}_to_{new_date_tag}_{target_language}.docx")

    return eval_results

def save_evaluation_report_to_docx(block_scores: list, faithfulness_score: float, df_eval: pd.DataFrame, output_filename: str):
    """
    Compiles block scores, faithfulness ratings, and translation summaries 
    into a structured, professional Word Document report card.
    """
    
    
    doc = docx.Document()
    
    # 1. Base Typography Styling
    style = doc.styles['Normal']
    font = style.font
    font.name = 'Arial'
    font.size = Pt(10.5)
    
    # Title Configuration
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    title_run = title.add_run("📊 PIPELINE EVALUATION REPORT CARD")
    title_run.font.size = Pt(18)
    title_run.bold = True
    title_run.font.color.rgb = RGBColor(0x1B, 0x36, 0x5D) # Navy Blue
    
    doc.add_paragraph(f"Report Generated: {datetime.now().strftime('%d %b %Y %H:%M:%S')}")
    doc.add_paragraph("").paragraph_format.space_after = Pt(12)
    
    # 2. Section 1: Core RAG Extraction Metrics
    doc.add_heading("1. Core RAG Extraction Metrics", level=1)
    
    # Highlighted Callout Panel for Core Faithfulness
    p_faith = doc.add_paragraph()
    p_faith.paragraph_format.left_indent = docx.shared.Inches(0.25)
    run_faith_label = p_faith.add_run("Overall Summary Faithfulness Score: ")
    run_faith_val = p_faith.add_run(f"{faithfulness_score:.2%}")
    run_faith_val.bold = True
    if faithfulness_score >= 0.95:
        run_faith_val.font.color.rgb = RGBColor(0x2E, 0x7D, 0x32) # Green Pass
    else:
        run_faith_val.font.color.rgb = RGBColor(0xC6, 0x28, 0x28) # Red Warning
        
    # Block-by-Block Granular Extraction Scores
    doc.add_heading("Granular Chunk-Level Extraction Performance", level=2)
    doc.add_paragraph(
        "The table below details the extraction accuracy score for each localized chunk block processed "
        "by the comparison engine prior to master document synthesis."
    )
    
    # Build a small table for block performance logs
    block_table = doc.add_table(rows=1, cols=2)
    block_table.style = 'Light Shading Accent 1'
    hdr_cells = block_table.rows[0].cells
    hdr_cells[0].text = 'Source Document Block ID'
    hdr_cells[1].text = 'Extraction Accuracy Quality Score'
    
    for i, score in enumerate(block_scores):
        row_cells = block_table.add_row().cells
        row_cells[0].text = f"Block {i + 1}"
        row_cells[1].text = f"{score:.2%}"
        
    doc.add_paragraph("").paragraph_format.space_after = Pt(18)
    
    # 3. Section 2: Multi-Language Translation Verification Summary
    doc.add_heading("2. Multi-Language Translation Verification Dashboard", level=1)
    doc.add_paragraph(
        "The following summary table aggregates reference-free cross-lingual quality estimation ratings, "
        "statutory tracking validations, and formatting layout constraints recorded across all translation threads."
    )
    
    # Dynamically build Word Table directly from Pandas DataFrame
    # Columns count matches df_eval widths shape
    eval_table = doc.add_table(rows=1, cols=len(df_eval.columns))
    eval_table.style = 'Light Shading Accent 1'
    
    # Map column headers
    for j, col_name in enumerate(df_eval.columns):
        eval_table.rows[0].cells[j].text = str(col_name).replace('_', ' ').title()
        
    # Populate cell arrays sequentially row-by-row
    for _, row in df_eval.iterrows():
        row_cells = eval_table.add_row().cells
        for j, val in enumerate(row):
            # Format display arrays safely into string lists
            if isinstance(val, list):
                row_cells[j].text = ", ".join(map(str, val)) if val else "None"
            elif isinstance(val, float):
                row_cells[j].text = f"{val:.2%}"
            else:
                row_cells[j].text = str(val)
                
    # Save completed document grid payload to disk
    doc.save(output_filename)
    print(f"\n✨ Evaluation Audit log document successfully compiled at: {os.path.abspath(output_filename)}")

In [11]:
with open("current_mining_act_file.pkl", "rb") as file:
    current_mining_act_file = pickle.load(file)
print(f"Current loaded mining act file: current_mining_act_file")

current_mining_act_file = "" # overwrite for testing

if current_mining_act_file == new_mining_act_file:
    print("No Change in mining Act policy")

else:
    print("New Changes detected.")
    print("-" * 60)

    client = genai.Client(api_key = GEMINI_API_KEY)
    print(f"\nStarting analysis across {max(len(old_chunks), len(new_chunks))} structural blocks...")
    print("-" * 60)

    respone_text, compare_new_old_chunks_block_scores = compare_new_old_chunks(new_chunks, 
                                          old_chunks, 
                                          new_mining_act_file,
                                          old_mining_act_file)
    
    master_response, new_date_tag, old_date_tag, faithfulness_rating= get_policy_changes(respone_text,
                                                                      new_mining_act_file,
                                                                      old_mining_act_file)
    all_eval_records = []

    print("\nStarting multi-language translation pipeline...")
    print("=" * 60)
    
    for i in list_of_target_language:
        eval_results = tranlate_master_respone(master_response,i,
                                new_date_tag, old_date_tag)
        eval_results["target_language"] = i
        all_eval_records.append(eval_results)
    
    df_eval_summary = pd.DataFrame(all_eval_records)
    
    cols = ['target_language', 'translation_quality_score', 'altered_statutory_codes', 'broken_markdown_structures', 'errors_detected']
    df_eval_summary = df_eval_summary[cols]
    display(df_eval_summary)
    
    with open("current_mining_act_file.pkl", "wb") as file:
        pickle.dump(new_mining_act_file, file)

    df_eval_summary = pd.DataFrame(all_eval_records)
    cols = ['target_language', 'translation_quality_score', 'altered_statutory_codes', 'broken_markdown_structures', 'errors_detected']
    df_eval_summary = df_eval_summary[cols]
    
    # 2. Assign a file save destination using your date tags
    eval_doc_filename = f"Pipeline_Evaluation_Audit_Report_{old_date_tag}_to_{new_date_tag}.docx"
    
    # 3. Call your compilation wrapper to commit metrics to disk
    save_evaluation_report_to_docx(
        block_scores=compare_new_old_chunks_block_scores,
        faithfulness_score=faithfulness_rating,
        df_eval=df_eval_summary,
        output_filename=eval_doc_filename
    )

Current loaded mining act file: current_mining_act_file
New Changes detected.
------------------------------------------------------------

Starting analysis across 7 structural blocks...
------------------------------------------------------------
Block 1: Text variation detected! Requesting localized analysis...

--- [DIFF REPORT FOR BLOCK 1] ---
### Legislative Data Analysis Report: Mining Act 1978

This report details the amendments observed between the older version (dated 09-p0-00, 9 Sep 2025) and the newer version (dated 09-q0-00, 28 May 2026) of the *Mining Act 1978*.

---

#### 1. Executive Summary
The changes between the two versions are **minimal and primarily technical/administrative**. The amendments focus on updating references to petroleum-related legislation and removing a redundant section. There are no substantive changes to the core regulatory framework, mining tenement processes, or administrative powers of the Minister.

---

#### 2. Specific Section Amendments

* 

# Informe de Auditoría de Reforma Legislativa: MINING ACT 1978
**Análisis Comparativo: Revisión 9_sep_2025 a Revisión 28_may_2026**

## 1. Resumen Ejecutivo
La transición entre las revisiones de septiembre de 2025 y mayo de 2026 de la *Mining Act 1978* se caracteriza por actualizaciones administrativas específicas y mejoras procedimentales. Las enmiendas se centran en la modernización de las referencias legislativas cruzadas a los estatutos de recursos energéticos, la formalización de los requisitos de solicitud para licencias diversas, la ampliación de la discrecionalidad ministerial respecto a las solicitudes de tenencia (*tenement applications*) y el fortalecimiento de las facultades de ejecución judicial dentro del *Warden’s Court*. El marco regulatorio permanece estable, sin cambios sustanciales en la arquitectura central de arrendamiento de tenencias.

## 2. Matriz Estatutaria Definitiva

### A. Adiciones Principales de Partes y Secciones
*   **Section 93 (Mapa que acompaña a la solicitud):** Formaliza el requisito de que las solicitudes de licencias diversas incluyan una descripción escrita y un mapa delimitado del área objeto de la solicitud.
*   **Section 94 (Términos y condiciones):** Establece el marco regulatorio para las condiciones de las licencias diversas, incluyendo la facultad discrecional del registrador de minería o del *warden* para imponer términos específicos y un mecanismo de apelación para los solicitantes respecto a condiciones consideradas "irrazonables".
*   **Section 111A (El Ministro puede rescindir o rechazar sumariamente ciertas solicitudes):** Otorga al Ministro la autoridad para rescindir solicitudes antes de la determinación del *warden* o rechazar sumariamente solicitudes por motivos de interés público (por ejemplo, preocupaciones por perturbación del terreno). También proporciona un mecanismo para que el Ministro conceda solicitudes de renovación tardía para arrendamientos mineros cuando el antiguo arrendatario haya observado sustancialmente los requisitos de la Ley.

### B. Disposiciones Derogadas, Omitidas y Suprimidas
*   **Section 8A (Derechos respecto a esquisto bituminoso o carbón):** Esta disposición fue derogada (No. 17 de 2024 s. 406), eliminando derechos específicos respecto al esquisto bituminoso o carbón para terrenos sujetos a permisos de exploración de petróleo.

### C. Secciones Alteradas y Modernizadas
*   **Section 8 (Términos utilizados):** Definiciones actualizadas para "minerales" y referencias a la *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* y a la *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982*.
*   **Section 139 (Desacato al tribunal):** Ampliada para incluir los subapartados (2), (3) y (4), codificando la autoridad del *warden* para imponer multas (hasta $1,000) y penas de prisión (hasta 14 días) por desacato, y estableciendo procedimientos para la ejecución de multas y el cumplimiento de órdenes.
*   **Section 140 (Sentencias, ejecución de):** Recién insertada para proporcionar una vía procedimental para la ejecución de las sentencias del *Warden’s Court* a través de tribunales de jurisdicción competente.
*   **Section 159 (Disputas entre licenciatarios y otras personas):** Actualizada para reflejar el título modernizado de la *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967*.
*   **Second Schedule (Disposiciones transitorias):** Se añadió la Cláusula 30, otorgando al Gobernador la facultad de dictar reglamentos transitorios, de salvaguardia o basados en la aplicación derivados de la *Mining Amendment (Transfer of Royalty Administration) Act 2025*.

## 3. Análisis Estructural y Temático

### A. Modificaciones Formales a la Terminología Jurídica

| Categoría / Contexto | Fraseología Legada | Fraseología Modernizada / Actualizada | Impacto Regulatorio Estratégico |
| :--- | :--- | :--- | :--- |
| **Legislación Energética** | *Petroleum and Geothermal Energy Resources Act 1967* | *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* | Asegura la consistencia estatutaria con los mandatos ampliados de almacenamiento de gases de efecto invernadero. |
| **Legislación Energética** | *Petroleum (Submerged Lands) Act 1982* | *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982* | Alinea las referencias regulatorias en alta mar con la legislación de almacenamiento actualizada. |

### B. Arquitectura de Cumplimiento, Acciones Punitivas y Disparadores de Caducidad
*   **Ejecución Judicial:** La introducción de la Section 139(2)–(4) y la Section 140 proporciona al *Warden’s Court* mecanismos punitivos y de ejecución explícitos. Estas adiciones hacen que el tribunal pase de una dependencia de poderes inherentes a un marco codificado para gestionar el desacato y ejecutar sentencias monetarias o no monetarias.
*   **Intervención Ministerial:** La Section 111A introduce un disparador administrativo de alto nivel que permite al Ministro eludir los procesos estándar de solicitud de tenencia, priorizando el interés público y la "observancia sustancial" de las condiciones de arrendamiento sobre la progresión procedimental estándar.

## 4. Metadatos Administrativos e Históricos
*   **Tabla de Compilación:** Actualizada para incluir la *Petroleum Legislation Amendment Act 2024* (No. 17 de 2024), señalando su entrada en vigor el 28 de mayo de 2026.
*   **Reglamentos Transitorios:** La Cláusula 30 del *Second Schedule* introduce una disposición de caducidad (*sunset provision*) de dos años para los reglamentos transitorios, permitiendo efectos retroactivos siempre que la fecha no sea anterior a la entrada en vigor de la *Mining Amendment (Transfer of Royalty Administration) Act 2025*.
*   **Metadatos del Documento:** La revisión de 2026 refleja la atribución de publicación actualizada (Impresor del Gobierno: Andrew Jones) y actualizaciones de derechos de autor.

🌐 Running translation quality checks for Spanish...
  └─ [RESULT] Quality Score: 98.00%
     📋 Logged Discrepancies: ["Minor stylistic inconsistency: The translation occasionally retains English terms like 'Warden's Court' and 'Section' in the body text, which is acceptable in specialized legal contexts but could be fully localized for a more formal Spanish legal report."]
Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Spanish.docx


# विधायी सुधार ऑडिट रिपोर्ट: MINING ACT 1978
**तुलनात्मक विश्लेषण: संशोधन 9_Sep_2025 से संशोधन 28_May_2026**

## 1. कार्यकारी सारांश
*Mining Act 1978* के सितंबर 2025 और मई 2026 के संशोधनों के बीच का संक्रमण लक्षित प्रशासनिक अद्यतनों और प्रक्रियात्मक संवर्द्धन द्वारा चिह्नित है। ये संशोधन ऊर्जा संसाधन संविधियों के विधायी क्रॉस-रेफरेंस के आधुनिकीकरण, विविध लाइसेंसों (miscellaneous licences) के लिए आवेदन आवश्यकताओं के औपचारिककरण, टेनेमेंट आवेदनों के संबंध में मंत्रिस्तरीय विवेकाधिकार के विस्तार, और वार्डन कोर्ट (Warden’s Court) के भीतर न्यायिक प्रवर्तन शक्तियों को मजबूत करने पर केंद्रित हैं। विनियामक ढांचा स्थिर बना हुआ है, जिसमें मुख्य टेनेमेंट लीजिंग संरचना में कोई ठोस बदलाव नहीं किया गया है।

## 2. निश्चित सांविधिक मैट्रिक्स

### A. प्रमुख भाग और धारा परिवर्धन
*   **धारा 93 (आवेदन के साथ संलग्न मानचित्र):** विविध लाइसेंस आवेदनों के लिए लिखित विवरण और विषय क्षेत्र का सीमांकित मानचित्र शामिल करने की आवश्यकता को औपचारिक रूप देती है।
*   **धारा 94 (नियम और शर्तें):** विविध लाइसेंस शर्तों के लिए विनियामक ढांचा स्थापित करती है, जिसमें विशिष्ट शर्तें लागू करने के लिए खनन रजिस्ट्रार या वार्डन की विवेकाधीन शक्ति और "अनुचित" शर्तों के संबंध में आवेदकों के लिए अपील तंत्र शामिल है।
*   **धारा 111A (मंत्री कुछ आवेदनों को समाप्त या संक्षिप्त रूप से अस्वीकार कर सकते हैं):** मंत्री को वार्डन के निर्णय से पहले आवेदनों को समाप्त करने या सार्वजनिक हित के आधार पर (जैसे, भूमि अशांति संबंधी चिंताएं) आवेदनों को संक्षिप्त रूप से अस्वीकार करने का अधिकार प्रदान करती है। यह मंत्री को उन खनन पट्टों के लिए विलंबित नवीनीकरण आवेदनों को स्वीकार करने का तंत्र भी प्रदान करती है जहां पूर्व पट्टेदार ने अधिनियम की आवश्यकताओं का पर्याप्त रूप से पालन किया है।

### B. निरसित, लोपित और विलोपित प्रावधान
*   **धारा 8A (ऑयल शेल या कोयले के संबंध में अधिकार):** इस प्रावधान को निरसित कर दिया गया (No. 17 of 2024 s. 406), जिससे पेट्रोलियम अन्वेषण परमिट के अधीन भूमि के लिए ऑयल शेल या कोयले से संबंधित विशिष्ट अधिकार समाप्त हो गए।

### C. परिवर्तित और आधुनिक धाराएं
*   **धारा 8 (प्रयुक्त शब्द):** "खनिजों" के लिए अद्यतन परिभाषाएं और *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* तथा *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982* के संदर्भ।
*   **धारा 139 (न्यायालय की अवमानना):** उपधारा (2), (3), और (4) को शामिल करने के लिए विस्तारित, जो अवमानना के लिए जुर्माना ($1,000 तक) और कारावास (14 दिन तक) लगाने के वार्डन के अधिकार को संहिताबद्ध करती है, और जुर्माना प्रवर्तन तथा आदेश निर्वहन के लिए प्रक्रियाएं स्थापित करती है।
*   **धारा 140 (निर्णय, प्रवर्तन):** सक्षम क्षेत्राधिकार वाले न्यायालयों के माध्यम से वार्डन कोर्ट के निर्णयों को लागू करने के लिए प्रक्रियात्मक मार्ग प्रदान करने हेतु नई जोड़ी गई।
*   **धारा 159 (लाइसेंसधारियों और अन्य व्यक्तियों के बीच विवाद):** *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* के आधुनिक शीर्षक को दर्शाने के लिए अद्यतन।
*   **द्वितीय अनुसूची (संक्रमणकालीन प्रावधान):** खंड 30 जोड़ा गया, जो गवर्नर को *Mining Amendment (Transfer of Royalty Administration) Act 2025* से उत्पन्न संक्रमणकालीन, बचत, या आवेदन-आधारित विनियम बनाने की शक्ति प्रदान करता है।

## 3. संरचनात्मक और विषयगत विश्लेषण

### A. कानूनी शब्दावली में औपचारिक संशोधन

| श्रेणी / संदर्भ | पूर्व शब्दावली | आधुनिक / अद्यतन शब्दावली | रणनीतिक विनियामक प्रभाव |
| :--- | :--- | :--- | :--- |
| **ऊर्जा विधान** | *Petroleum and Geothermal Energy Resources Act 1967* | *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* | विस्तारित ग्रीनहाउस गैस भंडारण जनादेश के साथ सांविधिक स्थिरता सुनिश्चित करता है। |
| **ऊर्जा विधान** | *Petroleum (Submerged Lands) Act 1982* | *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982* | अपतटीय विनियामक संदर्भों को अद्यतन भंडारण विधान के साथ संरेखित करता है। |

### B. अनुपालन संरचना, दंडात्मक कार्रवाई, और जब्ती ट्रिगर
*   **न्यायिक प्रवर्तन:** धारा 139(2)–(4) और धारा 140 का परिचय वार्डन कोर्ट को स्पष्ट दंडात्मक और प्रवर्तन तंत्र प्रदान करता है। ये परिवर्धन न्यायालय को निहित शक्तियों पर निर्भरता से हटाकर अवमानना के प्रबंधन और मौद्रिक या गैर-मौद्रिक निर्णयों को लागू करने के लिए एक संहिताबद्ध ढांचे में स्थानांतरित करते हैं।
*   **मंत्रिस्तरीय हस्तक्षेप:** धारा 111A एक उच्च-स्तरीय प्रशासनिक ट्रिगर पेश करती है जो मंत्री को मानक टेनेमेंट आवेदन प्रक्रियाओं को दरकिनार करने की अनुमति देती है, जो मानक प्रक्रियात्मक प्रगति के बजाय सार्वजनिक हित और पट्टे की शर्तों के "पर्याप्त पालन" को प्राथमिकता देती है।

## 4. प्रशासनिक और ऐतिहासिक मेटाडेटा
*   **संकलन तालिका:** *Petroleum Legislation Amendment Act 2024* (No. 17 of 2024) को शामिल करने के लिए अद्यतन, जो 28 मई 2026 से इसके प्रारंभ होने का उल्लेख करती है।
*   **संक्रमणकालीन विनियम:** द्वितीय अनुसूची का खंड 30 संक्रमणकालीन विनियमों के लिए दो साल का सनसेट प्रावधान पेश करता है, जो पूर्वव्यापी प्रभाव की अनुमति देता है, बशर्ते कि तिथि *Mining Amendment (Transfer of Royalty Administration) Act 2025* के प्रारंभ होने से पहले की न हो।
*   **दस्तावेज़ मेटाडेटा:** 2026 का संशोधन अद्यतन प्रकाशन श्रेय (सरकारी मुद्रक: एंड्रयू जोन्स) और कॉपीराइट अद्यतनों को दर्शाता है।

🌐 Running translation quality checks for Hindi...
  └─ [RESULT] Quality Score: 98.00%
Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Hindi.docx


...

🌐 Running translation quality checks for Italian...
  └─ [RESULT] Quality Score: 98.00%
Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Italian.docx


# Raport z audytu reformy legislacyjnej: MINING ACT 1978
**Analiza porównawcza: Wersja z 9 września 2025 r. a wersja z 28 maja 2026 r.**

## 1. Podsumowanie wykonawcze
Przejście między wersjami *Mining Act 1978* z września 2025 r. a maja 2026 r. charakteryzuje się ukierunkowanymi aktualizacjami administracyjnymi oraz usprawnieniami proceduralnymi. Nowelizacje koncentrują się na modernizacji odniesień legislacyjnych do ustaw dotyczących zasobów energetycznych, sformalizowaniu wymogów dotyczących wniosków o koncesje różnego rodzaju (miscellaneous licences), rozszerzeniu uznaniowości ministerialnej w odniesieniu do wniosków o nadanie tytułu prawnego do terenu (tenement applications) oraz wzmocnieniu uprawnień w zakresie egzekwowania prawa przez sąd Warden’s Court. Ramy regulacyjne pozostają stabilne, bez istotnych zmian w podstawowej architekturze dzierżawy terenów górniczych.

## 2. Definitywna matryca ustawowa

### A. Główne dodatki do części i sekcji
*   **Section 93 (Mapa załączana do wniosku):** Formalizuje wymóg, aby wnioski o koncesje różnego rodzaju zawierały opis pisemny oraz wyznaczoną mapę obszaru, którego dotyczy wniosek.
*   **Section 94 (Warunki):** Ustanawia ramy regulacyjne dla warunków koncesji różnego rodzaju, w tym uznaniowe uprawnienie rejestratora górniczego lub urzędnika (warden) do nakładania określonych warunków oraz mechanizm odwoławczy dla wnioskodawców w odniesieniu do warunków „nieuzasadnionych”.
*   **Section 111A (Minister może zakończyć lub w trybie uproszczonym odrzucić niektóre wnioski):** Przyznaje Ministrowi uprawnienie do zakończenia rozpatrywania wniosków przed wydaniem decyzji przez urzędnika (warden) lub do odrzucenia wniosków w trybie uproszczonym ze względu na interes publiczny (np. obawy dotyczące naruszenia gruntów). Zapewnia również Ministrowi mechanizm udzielania spóźnionych wniosków o przedłużenie dzierżaw górniczych, w przypadku gdy poprzedni dzierżawca w znacznym stopniu przestrzegał wymogów ustawy.

### B. Przepisy uchylone, pominięte i usunięte
*   **Section 8A (Prawa w odniesieniu do łupków bitumicznych lub węgla):** Przepis ten został uchylony (No. 17 of 2024 s. 406), co znosi szczególne prawa dotyczące łupków bitumicznych lub węgla w odniesieniu do gruntów objętych zezwoleniami na poszukiwanie ropy naftowej.

### C. Sekcje zmienione i zmodernizowane
*   **Section 8 (Użyte terminy):** Zaktualizowano definicje „minerałów” oraz odniesienia do *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* oraz *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982*.
*   **Section 139 (Obraza sądu):** Rozszerzono o ustępy (2), (3) i (4), kodyfikujące uprawnienie urzędnika (warden) do nakładania grzywien (do 1000 USD) i kary pozbawienia wolności (do 14 dni) za obrazę sądu, a także ustanawiające procedury egzekwowania grzywien i uchylania postanowień.
*   **Section 140 (Wyroki, egzekucja):** Nowo wstawiona sekcja zapewniająca ścieżkę proceduralną dla egzekwowania wyroków Warden’s Court za pośrednictwem sądów właściwych miejscowo i rzeczowo.
*   **Section 159 (Spory między koncesjonariuszami a innymi osobami):** Zaktualizowano w celu odzwierciedlenia zmodernizowanego tytułu *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967*.
*   **Second Schedule (Przepisy przejściowe):** Dodano klauzulę 30, przyznającą Gubernatorowi uprawnienie do stanowienia przepisów przejściowych, przepisów o zachowaniu praw lub przepisów opartych na zastosowaniu, wynikających z *Mining Amendment (Transfer of Royalty Administration) Act 2025*.

## 3. Analiza strukturalna i tematyczna

### A. Formalne modyfikacje terminologii prawnej

| Kategoria / Kontekst | Dawne sformułowanie | Zmodernizowane / Zaktualizowane sformułowanie | Strategiczny wpływ regulacyjny |
| :--- | :--- | :--- | :--- |
| **Ustawodawstwo energetyczne** | *Petroleum and Geothermal Energy Resources Act 1967* | *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* | Zapewnia spójność ustawową z rozszerzonymi mandatami dotyczącymi składowania gazów cieplarnianych. |
| **Ustawodawstwo energetyczne** | *Petroleum (Submerged Lands) Act 1982* | *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982* | Dostosowuje odniesienia do przepisów morskich do zaktualizowanego ustawodawstwa dotyczącego składowania. |

### B. Architektura zgodności, działania karne i przesłanki przepadku
*   **Egzekwowanie sądowe:** Wprowadzenie Section 139(2)–(4) oraz Section 140 zapewnia Warden’s Court wyraźne mechanizmy karne i egzekucyjne. Dodatki te przekształcają sąd z organu polegającego na uprawnieniach przyrodzonych w organ działający w oparciu o skodyfikowane ramy zarządzania obrazą sądu oraz egzekwowania wyroków pieniężnych i niepieniężnych.
*   **Interwencja ministerialna:** Section 111A wprowadza wysokopoziomowy wyzwalacz administracyjny, pozwalający Ministrowi na pominięcie standardowych procesów składania wniosków o nadanie tytułu prawnego do terenu, przedkładając interes publiczny i „istotne przestrzeganie” warunków dzierżawy nad standardowy przebieg procedury.

## 4. Metadane administracyjne i historyczne
*   **Tabela kompilacji:** Zaktualizowana o *Petroleum Legislation Amendment Act 2024* (No. 17 of 2024), z odnotowaniem daty wejścia w życie: 28 maja 2026 r.
*   **Przepisy przejściowe:** Klauzula 30 Second Schedule wprowadza dwuletnią klauzulę wygaśnięcia (sunset provision) dla przepisów przejściowych, umożliwiając działanie wsteczne pod warunkiem, że data nie przypada przed wejściem w życie *Mining Amendment (Transfer of Royalty Administration) Act 2025*.
*   **Metadane dokumentu:** Wersja z 2026 r. odzwierciedla zaktualizowane przypisanie publikacji (Government Printer: Andrew Jones) oraz aktualizacje praw autorskich.

🌐 Running translation quality checks for Polish...
  └─ [RESULT] Quality Score: 95.00%
     📋 Logged Discrepancies: ["In Section 139, the translation converted '$1,000' to '1000 USD', which is a minor deviation from the original currency format.", "The term 'Warden's Court' was left untranslated in some instances and translated as 'urzędnik (warden)' in others, creating slight inconsistency in terminology handling."]
Generating formatted report configuration...
Success! Document created at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Legislative_Reform_Audit_Report_9_Sep_2025_to_28_May_2026_Polish.docx


,target_language,translation_quality_score,altered_statutory_codes,broken_markdown_structures,errors_detected
0,Spanish,0.98,False,False,[Minor stylistic inconsistency: The translatio...
1,Hindi,0.98,False,False,[]
2,Tamil,0.98,False,False,[]
3,French,0.98,False,False,[]
4,German,0.98,False,False,[]
5,Italian,0.98,False,False,[]
6,Polish,0.95,False,False,"[In Section 139, the translation converted '$1..."



✨ Evaluation Audit log document successfully compiled at: /Users/daveng/Library/CloudStorage/GoogleDrive-davegn@gmail.com/My Drive/School/ICT619 Artificial Intelligence/Assignment 1/Pipeline_Evaluation_Audit_Report_9_Sep_2025_to_28_May_2026.docx


In [12]:
Markdown(master_response.text)

# Legislative Reform Audit Report: MINING ACT 1978
**Comparative Analysis: Revision 9_Sep_2025 to Revision 28_May_2026**

## 1. Executive Summary
The transition between the September 2025 and May 2026 revisions of the *Mining Act 1978* is characterized by targeted administrative updates and procedural enhancements. The amendments focus on the modernization of legislative cross-references to energy resource statutes, the formalization of application requirements for miscellaneous licences, the expansion of ministerial discretion regarding tenement applications, and the strengthening of judicial enforcement powers within the Warden’s Court. The regulatory framework remains stable, with no substantive shifts in the core tenement leasing architecture.

## 2. Definitive Statutory Matrix

### A. Major Part and Section Additions
*   **Section 93 (Map to accompany application):** Formalizes the requirement for miscellaneous licence applications to include a written description and a delineated map of the subject area.
*   **Section 94 (Terms and conditions):** Establishes the regulatory framework for miscellaneous licence conditions, including the discretionary power of the mining registrar or warden to impose specific terms and an appeal mechanism for applicants regarding "unreasonable" conditions.
*   **Section 111A (Minister may terminate or summarily refuse certain applications):** Grants the Minister authority to terminate applications prior to warden determination or summarily refuse applications based on public interest grounds (e.g., land disturbance concerns). It also provides a mechanism for the Minister to grant late renewal applications for mining leases where the former lessee has substantially observed Act requirements.

### B. Repealed, Omitted, and Deleted Provisions
*   **Section 8A (Rights in respect of oil shale or coal):** This provision was repealed (No. 17 of 2024 s. 406), removing specific rights regarding oil shale or coal for land subject to petroleum exploration permits.

### C. Altered and Modernized Sections
*   **Section 8 (Terms used):** Updated definitions for "minerals" and references to the *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* and the *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982*.
*   **Section 139 (Contempt of court):** Expanded to include subsections (2), (3), and (4), codifying the warden’s authority to impose fines (up to $1,000) and imprisonment (up to 14 days) for contempt, and establishing procedures for fine enforcement and order discharge.
*   **Section 140 (Judgments, enforcement of):** Newly inserted to provide a procedural pathway for enforcing warden’s court judgments through courts of competent jurisdiction.
*   **Section 159 (Disputes between licensees and other persons):** Updated to reflect the modernized title of the *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967*.
*   **Second Schedule (Transitional provisions):** Added Clause 30, granting the Governor power to make transitional, savings, or application-based regulations arising from the *Mining Amendment (Transfer of Royalty Administration) Act 2025*.

## 3. Structural and Thematic Analysis

### A. Formal Modifications to Legal Terminology

| Category / Context | Legacy Phrasing | Modernized / Updated Phrasing | Strategic Regulatory Impact |
| :--- | :--- | :--- | :--- |
| **Energy Legislation** | *Petroleum and Geothermal Energy Resources Act 1967* | *Petroleum, Geothermal Energy and Greenhouse Gas Storage Act 1967* | Ensures statutory consistency with expanded greenhouse gas storage mandates. |
| **Energy Legislation** | *Petroleum (Submerged Lands) Act 1982* | *Petroleum and Greenhouse Gas Storage (Submerged Lands) Act 1982* | Aligns offshore regulatory references with updated storage legislation. |

### B. Compliance Architecture, Punitive Actions, and Forfeiture Triggers
*   **Judicial Enforcement:** The introduction of Section 139(2)–(4) and Section 140 provides the Warden’s Court with explicit punitive and enforcement mechanisms. These additions transition the court from a reliance on inherent powers to a codified framework for managing contempt and enforcing monetary or non-monetary judgments.
*   **Ministerial Intervention:** Section 111A introduces a high-level administrative trigger allowing the Minister to bypass standard tenement application processes, prioritizing public interest and "substantial observance" of lease conditions over standard procedural progression.

## 4. Administrative and Historical Metadata
*   **Compilation Table:** Updated to include the *Petroleum Legislation Amendment Act 2024* (No. 17 of 2024), noting its commencement on 28 May 2026.
*   **Transitional Regulations:** Clause 30 of the Second Schedule introduces a two-year sunset provision for transitional regulations, allowing for retrospective effect provided the date is not prior to the commencement of the *Mining Amendment (Transfer of Royalty Administration) Act 2025*.
*   **Document Metadata:** The 2026 revision reflects updated publication attribution (Government Printer: Andrew Jones) and copyright updates.